# Dimensional — Product Dimension (SCD Type 1)

Build `globalmart.gold.dim_product` from bronze products with hash surrogate keys and MERGE upserts. Validates category translation conformance.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.product_dim as product_dim_module
importlib.reload(product_dim_module)

from src.dimensional.product_dim import (
    ProductDimConfig,
    build_product_dimension_source,
    apply_product_dimension_merge,
    validate_category_conformance,
    run_product_dimension,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = ProductDimConfig()
print("Loaded from:", product_dim_module.__file__)

In [ ]:
source = build_product_dimension_source(spark, config)
apply_product_dimension_merge(spark, source, config)
print("Products:", spark.table(config.target_table).count())
display(spark.table(config.target_table).limit(10))

In [ ]:
dim = spark.table(config.target_table)
_, conformance = validate_category_conformance(spark, dim, config)
print(conformance)

In [ ]:
import json

report = run_product_dimension(spark, config)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_product.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)